![Python Banner](../../Image/05_04_doubleML.png)


# 4.8 Double ML with Ensemble Learners

The orthogonal score does not care which algorithm fits $E[Y\mid X]$ and $E[D\mid X]$, as long as those fits are cross-fitted and converge fast enough. A pipeline, an average of several models, or a stack is just another function from $X$ to a prediction.

This notebook builds those learners with scikit-learn and plugs them into the partialling-out score. `PartiallyLinearDML` is that same score with the package's ridge and random-forest nuisances.


## Overview

| Learner | Nuisance |
|---------|----------|
| Lasso or a forest | One algorithm |
| Pipeline | Scale the columns, then fit |
| Averaging ensemble | Mean of lasso, a forest, and gradient boosting |
| Stack | First-stage predictions joined back to $X$, then a second model |
| Small grid | Choose the lasso penalty by cross-validated outcome error, then refit the score |

Changing a decent nuisance should move $\hat\theta$ only within sampling error.


## Implementation in Python

Two samples. A simulated partially linear model with a continuous treatment and true effect 0.8. And `Data/data_401k.csv`: binary eligibility `e401`, outcome `net_tfa`, capped at 4,000 rows. `fit_plr` cross-fits any scikit-learn estimator. A classifier is used for the propensity when the treatment is binary.


## Set Up


In [1]:
import subprocess
import sys
from pathlib import Path

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "pyproject.toml").is_file() and (cand / "src" / "pycausalml").is_dir():
            return cand
    raise FileNotFoundError(
        "PyCausalML repository was not found. Run this notebook from the repository."
    )

root = _repo_root()
data_dir = root / "Data"
if not data_dir.is_dir():
    raise FileNotFoundError(
        "The Data directory was not found. It should sit next to pyproject.toml."
    )

try:
    import pycausalml  # noqa: F401
    print("PyCausalML ready.")
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{root}[plot]"],
    )
    print("PyCausalML installed.")
print("Data directory:", data_dir)


PyCausalML ready.
Data directory: /home/zia207/Github/PyCausalML/Data


In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import Lasso, LassoCV, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    mean_squared_error,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from pycausalml import (
    CausalForestDML,
    InteractiveRegressionDML,
    KernelDML,
    LinearDML,
    NonParamDML,
    PartiallyLinearDML,
    SparseLinearDML,
    explain_cate,
)
from pycausalml.interpretation import plot_cate_hist

plt.rcParams["figure.dpi"] = 110
print("Imported PyCausalML")

from sklearn.base import clone
from sklearn.ensemble import StackingRegressor, VotingClassifier, VotingRegressor
from sklearn.tree import DecisionTreeRegressor


Imported PyCausalML


In [3]:
def fit_plr(X, d, y, learner_y, learner_d, n_splits=2, seed=0, discrete=False):
    y = np.asarray(y, dtype=float)
    d = np.asarray(d, dtype=float)
    y_hat = np.zeros(len(y))
    d_hat = np.zeros(len(y))
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, te in splitter.split(X):
        model_y, model_d = clone(learner_y), clone(learner_d)
        model_y.fit(X[tr], y[tr])
        y_hat[te] = model_y.predict(X[te])
        if discrete:
            model_d.fit(X[tr], d[tr].astype(int))
            d_hat[te] = model_d.predict_proba(X[te])[:, 1]
        else:
            model_d.fit(X[tr], d[tr])
            d_hat[te] = model_d.predict(X[te])
    y_res, d_res = y - y_hat, d - d_hat
    theta = float(np.dot(d_res, y_res) / np.dot(d_res, d_res))
    psi = d_res * (y_res - theta * d_res) / np.mean(d_res ** 2)
    se = float(np.std(psi, ddof=1) / np.sqrt(len(psi)))
    return theta, se


def simulate_plr(n=900, p=12, theta=0.8, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, p))
    beta = np.zeros(p)
    beta[:4] = np.array([0.8, -0.5, 0.4, 0.3])
    treatment = X @ beta + rng.normal(size=n)
    outcome = theta * treatment + np.sin(X[:, 0]) + 0.3 * X[:, 1] + rng.normal(size=n)
    return X, treatment, outcome


print("Ensemble helpers ready")


Ensemble helpers ready


## Part 1: One learner on the simulation


In [4]:
X, d, y = simulate_plr()
lasso = Lasso(alpha=0.02, max_iter=5000)
forest = RandomForestRegressor(n_estimators=40, max_depth=4, min_samples_leaf=5, random_state=0, n_jobs=1)
theta_lasso, se_lasso = fit_plr(X, d, y, lasso, lasso, seed=1)
theta_forest, se_forest = fit_plr(X, d, y, forest, forest, seed=1)
package = PartiallyLinearDML(discrete_treatment=False, n_splits=2, model_y="ridge", random_state=1)
package.fit(X, d, y)
print("Lasso", round(theta_lasso, 3), "se", round(se_lasso, 3))
print("Forest", round(theta_forest, 3), "se", round(se_forest, 3))
print("PartiallyLinearDML", package.average_treatment_effect())
print("True theta 0.8")


Lasso 0.839 se 0.034
Forest 0.752 se 0.031
PartiallyLinearDML {'estimate': 0.8252916354100195, 'std.err': 0.03412305192149705, 'ci_lower': 0.758411682601295, 'ci_upper': 0.892171588218744}
True theta 0.8


## Part 2: 401(k) eligibility


In [5]:
pension = pd.read_csv(data_dir / "data_401k.csv")
if len(pension) > 4000:
    pension = pension.sample(4000, random_state=4).reset_index(drop=True)
x_cols = ["age", "inc", "fsize", "educ", "db", "marr", "twoearn", "pira", "hown"]
Xp = pension[x_cols].to_numpy(dtype=float)
yp = pension["net_tfa"].to_numpy(dtype=float)
dp = pension["e401"].to_numpy(dtype=float)
rf_y = RandomForestRegressor(n_estimators=40, max_depth=5, min_samples_leaf=5, random_state=2, n_jobs=1)
rf_d = RandomForestClassifier(n_estimators=40, max_depth=5, min_samples_leaf=5, random_state=2, n_jobs=1)
theta_rf, se_rf = fit_plr(Xp, dp, yp, rf_y, rf_d, discrete=True, seed=2)
pkg = PartiallyLinearDML(discrete_treatment=True, n_splits=2, model_y="rf", n_estimators=40, random_state=2)
pkg.fit(Xp, dp, yp, feature_names=x_cols)
print("Forest score", round(theta_rf, 1), "se", round(se_rf, 1))
print("PartiallyLinearDML", pkg.average_treatment_effect())


Forest score 8075.4 se 1736.5
PartiallyLinearDML {'estimate': 7409.295011703511, 'std.err': 1866.097950362166, 'ci_lower': 3751.8102373696515, 'ci_upper': 11066.77978603737}


## Part 3: A pipeline


In [6]:
pipe = Pipeline([("scale", StandardScaler()), ("lasso", Lasso(alpha=0.02, max_iter=5000))])
theta_pipe, se_pipe = fit_plr(X, d, y, pipe, pipe, seed=3)
print("Scaled lasso", round(theta_pipe, 3), "se", round(se_pipe, 3))
print("Unscaled lasso", round(theta_lasso, 3))


Scaled lasso 0.801 se 0.034
Unscaled lasso 0.839


## Part 4: An average of learners


In [7]:
average_y = VotingRegressor([
    ("lasso", Lasso(alpha=0.02, max_iter=5000)),
    ("forest", RandomForestRegressor(n_estimators=30, max_depth=4, random_state=0, n_jobs=1)),
    ("gbr", GradientBoostingRegressor(n_estimators=30, max_depth=2, random_state=0)),
])
theta_avg, se_avg = fit_plr(X, d, y, average_y, clone(average_y), seed=4)
print("Averaged nuisances, simulation", round(theta_avg, 3), "se", round(se_avg, 3))
average_d = VotingClassifier([
    ("logit", Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000)),
    ])),
    ("forest", RandomForestClassifier(n_estimators=30, max_depth=4, random_state=0, n_jobs=1)),
    ("gbr", GradientBoostingClassifier(n_estimators=30, max_depth=2, random_state=0)),
], voting="soft")
theta_vote, se_vote = fit_plr(Xp, dp, yp, clone(average_y), average_d, discrete=True, seed=4)
print("Averaged nuisances, 401(k)", round(theta_vote, 1), "se", round(se_vote, 1))


Averaged nuisances, simulation 0.796 se 0.032


Averaged nuisances, 401(k) 7890.9 se 1608.5


## Part 5: A stack


In [8]:
stack = StackingRegressor(
    estimators=[
        ("tree", DecisionTreeRegressor(max_depth=3, random_state=0)),
        ("ridge", Ridge(alpha=1.0)),
    ],
    final_estimator=Ridge(alpha=1.0),
    passthrough=True,
)
theta_stack, se_stack = fit_plr(X, d, y, stack, Ridge(alpha=1.0), seed=5)
print("Stacked outcome nuisance", round(theta_stack, 3), "se", round(se_stack, 3))


Stacked outcome nuisance 0.824 se 0.034


## Part 6: A small penalty grid

Three lasso penalties. The column `outcome CV-MSE` is how well each penalty predicts $Y$ from $X$. The partialling-out estimate should stay near 0.8 across the grid.


In [9]:
rows = []
best_alpha, best_mse = None, np.inf
splitter = KFold(n_splits=3, shuffle=True, random_state=6)
for alpha in (0.001, 0.02, 0.2):
    mse = []
    for tr, te in splitter.split(X):
        model = Lasso(alpha=alpha, max_iter=6000)
        model.fit(X[tr], y[tr])
        mse.append(mean_squared_error(y[te], model.predict(X[te])))
    mse = float(np.mean(mse))
    learner = Lasso(alpha=alpha, max_iter=6000)
    theta, se = fit_plr(X, d, y, learner, clone(learner), seed=6)
    rows.append({"alpha": alpha, "outcome CV-MSE": round(mse, 4), "theta": round(theta, 3), "se": round(se, 3)})
    if mse < best_mse:
        best_alpha, best_mse = alpha, mse
print("Lowest outcome CV-MSE at alpha", best_alpha)
pd.DataFrame(rows)


Lowest outcome CV-MSE at alpha 0.02


,alpha,outcome CV-MSE,theta,se
0,0.001,1.7709,0.853,0.034
1,0.020,1.7450,0.853,0.034
2,0.200,1.7955,0.839,0.033


## Summary

Pipelines, averages, and stacks are nuisance functions. The causal step is still the partialling-out score, which `PartiallyLinearDML` implements for ridge and random forests. On the simulation the estimate stays near the true value 0.8. On the 401(k) extract the same score estimates the eligibility effect.

## Resources

- Chernozhukov, V., et al. (2018). Double/debiased machine learning for treatment and structural parameters. *The Econometrics Journal*, 21(1), C1–C68.
- Pedregosa, F., et al. (2011). Scikit-learn: machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.
